# ETL ICFES — carga multi-periodo con control de esquema

Este notebook consolida los resultados de Saber 11 de DataIcfes (archivos
`.txt` planos) para los periodos `2021-2` a `2025-2`, teniendo en cuenta que
**no todos los periodos traen exactamente las mismas columnas**.

**Este notebook debe vivir en la MISMA carpeta que tus archivos `.txt`**
(los que vienen nombrados `Examen_Saber_11_20212.txt`,
`Examen_Saber_11_20221.txt`, etc.). Si prefieres tenerlos en otra carpeta,
solo ajusta `RAW_DATA_DIR` en la siguiente celda.

Como cada archivo real pesa entre ~25 MB y ~420 MB (varios cientos de miles
de filas), el notebook **no carga todo en memoria de una sola vez**: detecta
encoding/separador leyendo solo una muestra, perfila el esquema leyendo solo
encabezados, y procesa/consolida los datos **por bloques (chunks)**.

El notebook hace 4 cosas:

1. **Detecta** encoding y separador de cada archivo, y lee solo sus
   encabezados (rápido, sin cargar los datos).
2. **Perfila el esquema** de cada periodo y construye la tabla de control
   `etl_schema_control` (periodo, número de columnas, columnas nuevas,
   columnas eliminadas, fecha de carga, estado `OK`/`CAMBIO`).
3. **Armoniza** los esquemas (calcula el esquema "maestro" = unión de todas
   las columnas) y consolida todos los periodos, por bloques, en un único
   archivo `output/dataset_consolidado.csv`.
4. **Perfila** el resultado: filas por periodo, % de nulos exacto por
   columna (calculado sobre todo el dataset), y estadísticas descriptivas
   sobre una muestra (para no tener que cargar los ~2.6 millones de filas
   completas en memoria solo para ver rangos y magnitudes).


## 0. Configuración

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import csv
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

# --- Periodos académicos a procesar (año-semestre) --------------------------
PERIODOS = [
    "2021-2", "2022-1", "2022-2", "2023-1", "2023-2",
    "2024-1", "2024-2", "2025-1", "2025-2",
]

# --- Rutas --------------------------------------------------------------
# El notebook debe estar en la misma carpeta que los .txt de DataIcfes.
RAW_DATA_DIR = Path(".")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Tamaño de bloque para leer/consolidar los archivos grandes por partes
CHUNK_SIZE = 100_000

# Filas de muestra por periodo para las estadísticas descriptivas de la
# sección 4.3 (no se usa para la tabla de control ni para el consolidado,
# que sí cubren el dataset completo)
FILAS_MUESTRA_POR_PERIODO = 20_000

# Si el auto-detector de separador se equivoca con algún periodo puntual,
# fuérzalo aquí, ej: SEPARADORES_MANUAL = {"2022-1": "|"}
SEPARADORES_MANUAL = {}

def periodo_a_nombre_archivo(periodo: str) -> str:
    """'2021-2' -> 'Examen_Saber_11_20212.txt' (patrón real de los archivos de DataIcfes)"""
    codigo = periodo.replace("-", "")
    return f"Examen_Saber_11_{codigo}.txt"

print(f"Periodos a procesar ({len(PERIODOS)}): {PERIODOS}")


## 1. Detección de encoding/separador y lectura de encabezados

Se detecta la codificación y el separador de cada archivo leyendo solo una
muestra (no el archivo completo), y luego se leen únicamente los
encabezados (`nrows=0`) para el perfilado de esquema — así la sección 1 y 2
son casi instantáneas sin importar qué tan pesado sea cada archivo.


In [ ]:
def _detectar_encoding(archivo: Path, muestra_bytes: int = 200_000) -> str:
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            with open(archivo, "r", encoding=enc) as f:
                f.read(muestra_bytes)
            return enc
        except UnicodeDecodeError:
            continue
    return "latin-1"  # último recurso: latin-1 casi nunca falla al leer

def _detectar_separador(archivo: Path, encoding: str) -> str:
    with open(archivo, "r", encoding=encoding, errors="replace") as f:
        muestra = f.read(65_536)
    try:
        return csv.Sniffer().sniff(muestra, delimiters="|;,\t¬").delimiter
    except csv.Error:
        primera_linea = muestra.splitlines()[0] if muestra else ""
        candidatos = ["|", ";", ",", "\t", "¬"]
        return max(candidatos, key=primera_linea.count)

def info_archivo_periodo(periodo: str) -> dict:
    """Localiza el archivo del periodo y detecta su encoding/separador."""
    archivo = RAW_DATA_DIR / periodo_a_nombre_archivo(periodo)
    if not archivo.exists():
        raise FileNotFoundError(
            f"No encontré '{archivo}'. Verifica que el notebook esté en la misma "
            f"carpeta que los .txt de DataIcfes, o ajusta RAW_DATA_DIR."
        )
    encoding = _detectar_encoding(archivo)
    separador = SEPARADORES_MANUAL.get(periodo) or _detectar_separador(archivo, encoding)
    return {"archivo": archivo, "encoding": encoding, "separador": separador}

info_periodos = {periodo: info_archivo_periodo(periodo) for periodo in PERIODOS}
for periodo, info in info_periodos.items():
    print(f"[{periodo}] {info['archivo'].name} -> encoding={info['encoding']}, separador={info['separador']!r}")


In [ ]:
def leer_columnas(periodo: str) -> list:
    info = info_periodos[periodo]
    encabezado = pd.read_csv(info["archivo"], sep=info["separador"], encoding=info["encoding"], nrows=0)
    return list(encabezado.columns)

columnas_por_periodo = {periodo: leer_columnas(periodo) for periodo in PERIODOS}
for periodo, cols in columnas_por_periodo.items():
    print(f"[{periodo}] {len(cols)} columnas")


## 2. Perfilado de esquema por periodo

Para cada periodo (en orden cronológico) se compara el conjunto de columnas
contra el periodo inmediatamente anterior:

- **`columnas_nuevas`**: columnas que aparecen ahora y no estaban antes.
- **`columnas_eliminadas`**: columnas que estaban antes y ya no están.
- **`estado`**: `OK` si el esquema no cambió respecto al periodo anterior,
  `CAMBIO` si hubo al menos una columna nueva o eliminada.

El primer periodo de la lista se toma como línea base (`OK`, sin
comparación posible).


In [ ]:
def construir_tabla_control(columnas_por_periodo: dict, periodos_ordenados: list) -> pd.DataFrame:
    filas = []
    columnas_anteriores = None
    for periodo in periodos_ordenados:
        columnas_actuales = set(columnas_por_periodo[periodo])

        if columnas_anteriores is None:
            nuevas, eliminadas, estado = set(), set(), "OK"
        else:
            nuevas = columnas_actuales - columnas_anteriores
            eliminadas = columnas_anteriores - columnas_actuales
            estado = "CAMBIO" if (nuevas or eliminadas) else "OK"

        filas.append({
            "periodo": periodo,
            "numero_columnas": len(columnas_actuales),
            "columnas_nuevas": ", ".join(sorted(nuevas)) if nuevas else "-",
            "columnas_eliminadas": ", ".join(sorted(eliminadas)) if eliminadas else "-",
            "fecha_carga": datetime.now().strftime("%Y-%m-%d %H:%M"),
            "estado": estado,
        })
        columnas_anteriores = columnas_actuales

    return pd.DataFrame(filas)

etl_schema_control = construir_tabla_control(columnas_por_periodo, PERIODOS)
etl_schema_control


In [ ]:
# Vista resumida, al estilo "periodo | numero_columnas | estado"
etl_schema_control[["periodo", "numero_columnas", "estado"]]


## 3. Armonización de esquemas

Se calcula un **esquema maestro** (la unión de todas las columnas vistas en
cualquier periodo, preservando el orden en que aparecieron por primera vez).
Cada periodo se "reindexa" contra ese esquema maestro más adelante (sección
4), rellenando con `NaN` las columnas que no existían en ese periodo —así
se puede consolidar todo sin perder información ni romper nada, aunque los
periodos no compartan exactamente las mismas columnas.


In [ ]:
def calcular_esquema_maestro(columnas_por_periodo: dict, periodos_ordenados: list) -> list:
    maestro = []
    vistas = set()
    for periodo in periodos_ordenados:
        for col in columnas_por_periodo[periodo]:
            if col not in vistas:
                maestro.append(col)
                vistas.add(col)
    return maestro

esquema_maestro = calcular_esquema_maestro(columnas_por_periodo, PERIODOS)
print(f"Columnas en el esquema maestro: {len(esquema_maestro)}")


### 3.1 Matriz de cobertura de columnas por periodo

Muestra, para cada periodo, qué columnas del esquema maestro sí estaban
presentes originalmente (útil para detectar visualmente en qué momentos
cambió el esquema).


In [ ]:
cobertura = pd.DataFrame(
    {periodo: [col in columnas_por_periodo[periodo] for col in esquema_maestro] for periodo in PERIODOS},
    index=esquema_maestro,
).T  # filas = periodo, columnas = variable del esquema maestro

cobertura.replace({True: "x", False: ""})


## 4. Consolidación y perfilado por bloques

Aquí sí se leen los datos (no solo encabezados), pero **por bloques de
`CHUNK_SIZE` filas** para no cargar los ~2.6 millones de registros
completos en memoria de una sola vez. Cada bloque se reindexa contra el
esquema maestro (los archivos de DataIcfes ya traen su propia columna
`periodo`, aquí solo se estandariza su valor al formato `AAAA-S`) y se va
anexando al archivo `output/dataset_consolidado.csv`. De paso, se acumula
el conteo exacto de filas y de nulos por columna.


In [ ]:
ruta_consolidado = OUTPUT_DIR / "dataset_consolidado.csv"
if ruta_consolidado.exists():
    ruta_consolidado.unlink()  # empezar limpio en cada corrida

filas_por_periodo = {}
nulos_por_columna = pd.Series(0, index=esquema_maestro, dtype="int64")
primer_bloque = True

for periodo in PERIODOS:
    info = info_periodos[periodo]
    total_filas_periodo = 0
    lector = pd.read_csv(
        info["archivo"], sep=info["separador"], encoding=info["encoding"],
        chunksize=CHUNK_SIZE,
    )
    for bloque in lector:
        bloque = bloque.reindex(columns=esquema_maestro)
        bloque["periodo"] = periodo  # estandariza el valor (ya viene en el archivo, pero en formato distinto, ej. 20212)
        nulos_por_columna += bloque[esquema_maestro].isna().sum()
        bloque.to_csv(ruta_consolidado, mode="a", index=False, header=primer_bloque)
        primer_bloque = False
        total_filas_periodo += len(bloque)
    filas_por_periodo[periodo] = total_filas_periodo
    print(f"[{periodo}] {total_filas_periodo:,} filas procesadas y agregadas al consolidado")

total_filas = sum(filas_por_periodo.values())
print(f"\nTotal de filas consolidadas: {total_filas:,}")
print(f"Consolidado guardado en: {ruta_consolidado}")


### 4.1 Registros por periodo

In [ ]:
pd.Series(filas_por_periodo, name="filas").reindex(PERIODOS).to_frame()


### 4.2 % de nulos por columna

Calculado de forma exacta sobre el dataset completo (acumulado bloque a
bloque en la celda anterior, no sobre una muestra).


In [ ]:
pct_nulos = (nulos_por_columna / total_filas * 100).sort_values(ascending=False)
pct_nulos.round(1).to_frame("% nulos")


### 4.3 Estadísticas descriptivas (sobre una muestra)

Con ~2.6 millones de filas en total, calcular `describe()` sobre el dataset
completo es costoso y no siempre necesario para una primera mirada. Esta
celda toma solo las primeras `FILAS_MUESTRA_POR_PERIODO` filas de cada
periodo (no todo el dataset) para dar una idea rápida de rangos y
magnitudes. Para un análisis exhaustivo sobre el dataset completo, lee
`output/dataset_consolidado.csv` con algo pensado para datasets grandes
(Dask, Polars, DuckDB, o una base de datos) en vez de pandas puro.


In [ ]:
muestras = []
for periodo in PERIODOS:
    info = info_periodos[periodo]
    m = pd.read_csv(info["archivo"], sep=info["separador"], encoding=info["encoding"], nrows=FILAS_MUESTRA_POR_PERIODO)
    m = m.reindex(columns=esquema_maestro)
    m["periodo"] = periodo
    muestras.append(m)

df_muestra = pd.concat(muestras, ignore_index=True)
print(f"Muestra total: {df_muestra.shape[0]:,} filas, {df_muestra.shape[1]} columnas")
df_muestra.select_dtypes(include=["number"]).describe().T


In [ ]:
df_muestra.dtypes.to_frame("dtype")


## 4.4 Perfilado de calidad adicional (soporta el punto 7 del proyecto)

Chequeos puntuales que complementan el perfilado general de arriba: duplicados,
calidad de `estu_genero` (usado en R5), valores atípicos en `punt_global`,
nulos en los atributos de colegio (usados en R1-R4), e inconsistencias de
categorías en `cole_area_ubicacion`.


**¿Por qué comparar por `estu_consecutivo` y no por fila completa?** Lo que
se valida aquí no es "¿hay dos filas 100% idénticas en las 92 columnas?"
(duplicado exacto), sino algo más específico para este proyecto: ¿se repite
el valor de la **llave natural del grano** declarado en el punto 10 (un
estudiante-intento-periodo)? Si dos filas comparten `estu_consecutivo` pero
difieren en algún otro campo (ej. un registro corregido/retransmitido), un
chequeo de fila completa NO lo detectaría como duplicado — pero para el
modelo dimensional sí es un problema grave, porque no se sabría cuál de los
dos valores es el correcto para esa llave antes de generar la surrogate key
de la tabla de hechos.


In [ ]:
# --- Duplicados por estu_consecutivo, por periodo ---
print("=== Duplicados por estu_consecutivo ===")
for periodo in PERIODOS:
    info = info_periodos[periodo]
    d = pd.read_csv(info["archivo"], sep=info["separador"], encoding=info["encoding"], usecols=["estu_consecutivo"])
    dup = d["estu_consecutivo"].duplicated().sum()
    print(f"{periodo}: {len(d):>7,} filas, {dup} consecutivos duplicados")


In [ ]:
# --- Calidad de estu_genero (usado en R5) ---
partes = []
for periodo in PERIODOS:
    info = info_periodos[periodo]
    d = pd.read_csv(info["archivo"], sep=info["separador"], encoding=info["encoding"], usecols=["estu_genero"])
    partes.append(d)
genero = pd.concat(partes, ignore_index=True)
print("=== estu_genero ===")
print(genero["estu_genero"].value_counts(dropna=False))
print(f"% nulos: {genero['estu_genero'].isna().mean():.4%}")


In [ ]:
# --- Valores atípicos en punt_global (rango válido ICFES: 0-500) ---
partes = []
for periodo in PERIODOS:
    info = info_periodos[periodo]
    d = pd.read_csv(info["archivo"], sep=info["separador"], encoding=info["encoding"], usecols=["punt_global"], low_memory=False)
    partes.append(d)
pg = pd.concat(partes, ignore_index=True)["punt_global"]
print("=== punt_global ===")
print(f"min={pg.min()}, max={pg.max()}, nulos={pg.isna().sum()}")
print(f"registros con punt_global == 0 (posibles pruebas anuladas/ausentes): {(pg == 0).sum()}")
print(f"registros fuera de [0, 500]: {((pg < 0) | (pg > 500)).sum()}")


In [ ]:
# --- Nulos en los atributos de colegio (usados en R1-R4) ---
cols_colegio = ["cole_naturaleza", "cole_jornada", "cole_area_ubicacion", "cole_depto_ubicacion", "cole_nombre_establecimiento"]
partes = []
for periodo in PERIODOS:
    info = info_periodos[periodo]
    d = pd.read_csv(info["archivo"], sep=info["separador"], encoding=info["encoding"], usecols=cols_colegio, low_memory=False)
    partes.append(d)
colegio = pd.concat(partes, ignore_index=True)

print("=== Nulos en atributos de colegio (afectan R1-R4) ===")
for c in cols_colegio:
    print(f"{c}: {colegio[c].isna().sum():,} nulos ({colegio[c].isna().mean():.2%})")

# ¿Son las mismas filas las que quedan nulas en las 4 columnas? (indica "sin colegio", no error de captura)
mismas_filas = (colegio["cole_naturaleza"].isna() == colegio["cole_nombre_establecimiento"].isna()).mean()
print(f"\n% de coincidencia entre nulos de cole_naturaleza y cole_nombre_establecimiento: {mismas_filas:.2%}")
print("(cercano a 100% confirma que son estudiantes sin colegio asociado -- ej. validantes -- y no un error de captura)")


In [ ]:
# --- Inconsistencia de categorías en cole_area_ubicacion ---
partes = []
for periodo in PERIODOS:
    info = info_periodos[periodo]
    d = pd.read_csv(info["archivo"], sep=info["separador"], encoding=info["encoding"], usecols=["cole_area_ubicacion"], low_memory=False)
    partes.append(d)
area = pd.concat(partes, ignore_index=True)
print("=== Valores de cole_area_ubicacion (ojo: URBANA / URBANO son la misma categoría mal armonizada) ===")
print(area["cole_area_ubicacion"].value_counts(dropna=False))


## 5. Exportar tabla de control

In [ ]:
ruta_control = OUTPUT_DIR / "etl_schema_control.csv"
etl_schema_control.to_csv(ruta_control, index=False)
print(f"Guardado: {ruta_control}")
print(f"(El dataset consolidado ya quedó guardado en la sección 4: {ruta_consolidado})")


## 6. (Opcional) Automatizar la descarga vía `datos.gov.co`

Como ya tienes los `.txt` descargados manualmente desde DataIcfes, esta
sección es opcional — solo sirve si más adelante quieres automatizar la
descarga en vez de bajar cada periodo a mano. Algunos de estos datasets
también están publicados como Socrata en el portal de datos abiertos y se
pueden descargar directo como CSV/JSON así:

```
https://www.datos.gov.co/resource/{resource_id}.csv?$limit=50000&$offset=0
```

Como estos datasets suelen tener cientos de miles de filas, hay que paginar
con `$limit`/`$offset`, e idealmente usar un **app token** gratuito de
Socrata (te evita los límites de tasa más estrictos). Ejemplo de función
para complementar el flujo de este notebook:

```python
import requests

RESOURCE_IDS = {
    "2021-2": "xxxx-xxxx",
    "2022-1": "xxxx-xxxx",
    # ...
}
APP_TOKEN = None  # tu token de Socrata, opcional pero recomendado

def descargar_periodo_socrata(periodo, resource_id, limite=50000):
    base_url = f"https://www.datos.gov.co/resource/{resource_id}.csv"
    headers = {"X-App-Token": APP_TOKEN} if APP_TOKEN else {}
    partes, offset = [], 0
    while True:
        params = {"$limit": limite, "$offset": offset}
        resp = requests.get(base_url, params=params, headers=headers)
        resp.raise_for_status()
        chunk = pd.read_csv(io.StringIO(resp.text))
        if chunk.empty:
            break
        partes.append(chunk)
        offset += limite
    return pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()
```

Si automatizas la descarga, guarda cada respuesta como
`Examen_Saber_11_{periodo_sin_guion}.txt` en la misma carpeta del notebook
y el resto del flujo (secciones 1 a 5) funciona sin cambios.
